In [13]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import orca

from mazsim import data_loader, variable_loader

In [14]:
project_dir = Path.cwd().parent
orca.add_injectable("project_dir", project_dir)
orca.run(['load_settings', 'load_data', 'build_networks','register_variables'])

Running step 'load_settings'
Registered injectable: data_dir: data
Registered injectable: output_dir: output
Registered injectable: data_model: data_model.py
Registered injectable: geography_id: block_id
Registered injectable: base_year: 2020
Registered injectable: end_year: 2050
Registered injectable: calibrated: True
Registered injectable: hh_ct_type: subregional
Registered injectable: job_ct_type: subregional
Registered injectable: housing_vacancy_rate: 0.05
Registered injectable: output_tables: ['households', 'persons', 'jobs', 'housing_units']
Registered injectable: custom_steps_dir: configs
Registered injectable: custom_steps_files: ['custom_steps.py']
Registered injectable: custom_variables_dir: configs
Registered injectable: custom_variables_files: ['custom_variables.py']
Registered injectable: submodel_groups: {'hlcm': {'ct_type': 'hh_ct_type'}, 'hulcm': {'ct_type': 'hh_ct_type', 'unsegmented_injectable': 'reg_hulcm_step_names'}, 'jlcm': {'ct_type': 'job_ct_type', 'unsegmented

In [15]:
from urbansim.models import util
from urbansim_templates import modelmanager as mm
from urbansim_templates.models import LargeMultinomialLogitStep

from urbansim.models.util import (columns_in_filters, columns_in_formula)
from choicemodels.tools import MergedChoiceTable

mm.initialize(Path.joinpath(project_dir, "configs"))

No files from ModelManager 0.1.dev8 or later found in path 'c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs'


In [16]:
import matplotlib.pyplot as plt
%matplotlib notebook
%matplotlib inline

import seaborn as sb

from bokeh.io import output_notebook#, show
# from bokeh.plotting import Figure
# from datashader.bokeh_ext import create_ramp_legend, create_categorical_legend

output_notebook()

import datashader.transfer_functions as tf

import datashader as ds
from datashader.colors import viridis

def visualize_variable(variable_name):
    p = orca.get_table('blocks').to_frame(['x', 'y', variable_name])
    
    cvs = ds.Canvas(plot_width=1000, plot_height=700)
    agg = cvs.points(p, 'x', 'y', ds.mean(variable_name))
    img = tf.set_background(tf.shade(agg, cmap=viridis),"white")
    return img

def summed_probas(sum_variable, probas):
    p = orca.get_table('blocks').to_frame([sum_variable])
    p['proba'] = probas

    summed_puma = p.groupby(sum_variable).proba.sum()

    (summed_puma / summed_puma.sum()).plot(kind='bar')

def plot_probas(proba):
    p = orca.get_table('blocks').to_frame(['x', 'y'])
    p['proba'] = proba

    cvs = ds.Canvas(plot_width=1000, plot_height=700)
    agg = cvs.points(p, 'x', 'y', ds.mean('proba'))
    img= tf.set_background(tf.shade(agg, cmap=viridis),"white")
    return img

def corr_plot(selected_variables):
    cols = []
    for col in selected_variables:    
        if col.startswith('np'):
            cols.append(col.split('(')[-1][:-1])
        else:
            cols.append(col)

    X = orca.get_table('blocks').to_frame(cols)

    plt.subplots(figsize=(12, 12))
    sb.heatmap(X.corr(), annot=True, cmap="RdYlGn")
    plt.show()
    
def skew_plot(selected_variables):
    cols = []
    for col in selected_variables:    
        if col.startswith('np'):
            cols.append(col.split('(')[-1][:-1])
        else:
            cols.append(col)
            
    X = orca.get_table('blocks').to_frame(cols)
    X.skew().plot(kind='bar')

def create_probs_table(m, rep_chooser_filter = None, sample_alts= None, sample_choosers= None,idx='building_id'):
    filter_cols = columns_in_filters(to_str(m.chooser_filters) + " " + to_str(m.alt_filters) + to_str(rep_chooser_filter))
    colnames = columns_in_formula(m.model_expression) + columns_in_filters(filter_cols)
    alts = orca.get_table(m.alternatives).to_frame(colnames)
    alts = alts.query(to_str(m.alt_filters)) if m.alt_filters else alts
    obs = orca.get_table(m.choosers).to_frame(colnames)
    obs = obs.query(to_str(m.chooser_filters)) if m.chooser_filters else obs
    if idx in obs.columns:
        obs = obs.drop(columns=[idx])
    # Filter representative chooser based on rep_chooser_filter input
    if rep_chooser_filter:
        # check if rep_filter has columns from the interaction terms
        if (all(x in obs.columns for x in columns_in_filters(to_str(rep_chooser_filter)))):
            obs = obs.query(to_str(rep_chooser_filter))
        else:
            chooser_vars = [x for x in columns_in_formula(m.model_expression) if x in orca.get_table(m.choosers).columns]
            raise ValueError("Representative choosers' filters must be on these columns:\n *{}".format('\n *'.join(chooser_vars)))
    if sample_choosers:
        if sample_choosers <= len(obs):
            obs = obs.sample(n= sample_choosers)
        else:
            obs
    if len(obs) == 0:
        raise ValueError('No choosers left after filtering.')
    if len(alts) == 0:
        raise ValueError('No alternatives left after filtering.')
    
    if not sample_alts:
        sample_alts = len(alts)
    mtc = MergedChoiceTable(obs, alts, sample_size= sample_alts)
    probas = m.model.probabilities(mtc)
    probas = probas.reset_index(level=1).groupby(idx).sum()
    return probas

def to_str(w):
    if isinstance(w, str):
        return w
    if isinstance(w, list):
        return ' & '.join(w) if len(w) > 1 else w[0]
    if not w:
        return " "

Loading BokehJS ...

In [17]:
# Set explanatory variables
expl_vars = [
    'major_road_node_sum_800_flat',
    'st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time',
    'st_prop_aggr_sector_id_5_ave_1600_flat', 
    'st_ln_block_groups_mean_res_value', 
    'st_ln_block_groups_mean_res_rent',
    'st_ln_block_groups_density_housing_units', 
    'st_sum_income_ave_1600_flat', 
    'st_is_transit_stop_sum_1600_flat',
    'st_total_households_sum_2000_linear'
]

In [18]:
# jlcm1 Agriculture, Mining, Utilities & Construction
m = LargeMultinomialLogitStep()
m.choosers = ['jobs']
m.chooser_sample_size = 5000
m.chooser_filters = 'aggr_sector_id == 1'
m.alternatives = ['blocks']
m.choice_column = 'block_id'
m.constrained_choices = True
m.alt_sample_size = 100 
m.out_chooser_filters = '(block_id == "-1") & (aggr_sector_id == 1)'
m.alt_capacity = 'vacant_job_spaces'
m.out_alt_filters = 'vacant_job_spaces > 0'
m.model_expression = util.str_model_expression(expl_vars, add_constant=False)
m.fit()
m.name = 'jlcm1'

Disaggregating mean_res_value to blocks from block_groups
Calculating mean_res_value of blocks for block_groups
Disaggregating total_jobs_15_minutes_am_single_vehicle_to_work_travel_time to blocks from zones
Calculating number of jobs for zones
Disaggregating zone_id to jobs from blocks
Calculating sum_income of households for blocks
Calculating proportion aggr_sector_id 5 for blocks
Calculating number of jobs for blocks
Calculating number of households for blocks
Disaggregating density_housing_units to blocks from block_groups
Calculating density of housing_units for block_groups
Calculating number of housing_units for block_groups
Disaggregating block_group_id to housing_units from blocks
Calculating sum_acres of blocks for block_groups
Disaggregating mean_res_rent to blocks from block_groups
Calculating mean_res_rent of blocks for block_groups
                  CHOICEMODELS ESTIMATION RESULTS                  
Dep. Var.:                chosen   No. Observations:          5,000
Model

In [19]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'jlcm1.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'jlcm1'


In [20]:
# jlcm2 Manufacturing, Wholesale Trade, Transportation & Warehousing
m = LargeMultinomialLogitStep()
m.choosers = ['jobs']
m.chooser_sample_size = 5000
m.chooser_filters = 'aggr_sector_id == 2'
m.alternatives = ['blocks']
m.choice_column = 'block_id'
m.constrained_choices = True
m.alt_sample_size = 100 
m.out_chooser_filters = '(block_id == "-1") & (aggr_sector_id == 2)'
m.alt_capacity = 'vacant_job_spaces'
m.out_alt_filters = 'vacant_job_spaces > 0'
m.model_expression = util.str_model_expression(expl_vars, add_constant=False)
m.fit()
m.name = 'jlcm2'

                  CHOICEMODELS ESTIMATION RESULTS                  
Dep. Var.:                chosen   No. Observations:          5,000
Model:         Multinomial Logit   Df Residuals:              4,991
Method:       Maximum Likelihood   Df Model:                      9
Date:                 2026-09-16   Pseudo R-squ.:             0.231
Time:                      12:55   Pseudo R-bar-squ.:         0.231
AIC:                  35,425.667   Log-Likelihood:      -17,703.834
BIC:                  35,484.322   LL-Null:             -23,025.851
                                                                             coef   std err         z     P>|z|   Conf. Int.
----------------------------------------------------------------------------------------------------------------------------
major_road_node_sum_800_flat                                               0.0944     0.017     5.401     0.000             
st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time    2.4627

In [21]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'jlcm2.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'jlcm2'


In [22]:
# jlcm3 Retail Trade, Arts/Entertainment/Recreation, Accommodation & Food Services
m = LargeMultinomialLogitStep()
m.choosers = ['jobs']
m.chooser_sample_size = 5000
m.chooser_filters = 'aggr_sector_id == 3'
m.alternatives = ['blocks']
m.choice_column = 'block_id'
m.constrained_choices = True
m.alt_sample_size = 100 
m.out_chooser_filters = '(block_id == "-1") & (aggr_sector_id == 3)'
m.alt_capacity = 'vacant_job_spaces'
m.out_alt_filters = 'vacant_job_spaces > 0'
m.model_expression = util.str_model_expression(expl_vars, add_constant=False)
m.fit()
m.name = 'jlcm3'

                  CHOICEMODELS ESTIMATION RESULTS                  
Dep. Var.:                chosen   No. Observations:          5,000
Model:         Multinomial Logit   Df Residuals:              4,991
Method:       Maximum Likelihood   Df Model:                      9
Date:                 2026-09-16   Pseudo R-squ.:             0.110
Time:                      12:55   Pseudo R-bar-squ.:         0.110
AIC:                  40,993.642   Log-Likelihood:      -20,487.821
BIC:                  41,052.297   LL-Null:             -23,025.851
                                                                             coef   std err         z     P>|z|   Conf. Int.
----------------------------------------------------------------------------------------------------------------------------
major_road_node_sum_800_flat                                              -0.0132     0.013    -1.026     0.305             
st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time    1.3782

In [23]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'jlcm3.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'jlcm3'


In [24]:
# jlcm4 Information, Finance, Real Estate, Professional/Management/Admin Services
m = LargeMultinomialLogitStep()
m.choosers = ['jobs']
m.chooser_sample_size = 5000
m.chooser_filters = 'aggr_sector_id == 4'
m.alternatives = ['blocks']
m.choice_column = 'block_id'
m.constrained_choices = True
m.alt_sample_size = 100 
m.out_chooser_filters = '(block_id == "-1") & (aggr_sector_id == 4)'
m.alt_capacity = 'vacant_job_spaces'
m.out_alt_filters = 'vacant_job_spaces > 0'
m.model_expression = util.str_model_expression(expl_vars, add_constant=False)
m.fit()
m.name = 'jlcm4'

                  CHOICEMODELS ESTIMATION RESULTS                  
Dep. Var.:                chosen   No. Observations:          5,000
Model:         Multinomial Logit   Df Residuals:              4,991
Method:       Maximum Likelihood   Df Model:                      9
Date:                 2026-09-16   Pseudo R-squ.:             0.234
Time:                      12:55   Pseudo R-bar-squ.:         0.233
AIC:                  35,310.321   Log-Likelihood:      -17,646.160
BIC:                  35,368.975   LL-Null:             -23,025.851
                                                                             coef   std err         z     P>|z|   Conf. Int.
----------------------------------------------------------------------------------------------------------------------------
major_road_node_sum_800_flat                                               0.2859     0.015    19.599     0.000             
st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time    1.9919

In [25]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'jlcm4.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'jlcm4'


In [26]:
# jlcm5 Educational Services, Health Care, Government
m = LargeMultinomialLogitStep()
m.choosers = ['jobs']
m.chooser_sample_size = 5000
m.chooser_filters = 'aggr_sector_id == 5'
m.alternatives = ['blocks']
m.choice_column = 'block_id'
m.constrained_choices = True
m.alt_sample_size = 100 
m.out_chooser_filters = '(block_id == "-1") & (aggr_sector_id == 5)'
m.alt_capacity = 'vacant_job_spaces'
m.out_alt_filters = 'vacant_job_spaces > 0'
m.model_expression = util.str_model_expression(expl_vars, add_constant=False)
m.fit()
m.name = 'jlcm5'

                  CHOICEMODELS ESTIMATION RESULTS                  
Dep. Var.:                chosen   No. Observations:          5,000
Model:         Multinomial Logit   Df Residuals:              4,991
Method:       Maximum Likelihood   Df Model:                      9
Date:                 2026-09-16   Pseudo R-squ.:             0.185
Time:                      12:55   Pseudo R-bar-squ.:         0.184
AIC:                  37,555.363   Log-Likelihood:      -18,768.682
BIC:                  37,614.018   LL-Null:             -23,025.851
                                                                             coef   std err         z     P>|z|   Conf. Int.
----------------------------------------------------------------------------------------------------------------------------
major_road_node_sum_800_flat                                               0.1975     0.015    12.996     0.000             
st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time    0.8551

In [27]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'jlcm5.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'jlcm5'
